In [ ]:
# Import packages, load the study configuration and the CHIP tables written by
# 11_chip_tables
from pathlib import Path

from SSc_Somatic import configure, load_config, load_study
from SSc_Somatic.chip import analysis as chip_funcs
from SSc_Somatic.chip import plots as chip_plots_funcs

# Study configuration. Point this at another configuration file to analyze a different cohort.
config = configure(load_config(Path('..') / 'config' / 'example_SSc.yaml'))
data = load_study(config, 'chip')

# Cohort labels used below
status_col = config.require('cohort', 'status_column')
case_label = config.require('cohort', 'case_label')
control_label = config.require('cohort', 'control_label')

# Tables written by 11_chip_tables
wl_final_anno_df = data.whitelist_calls
wl_final_merged_df = data.whitelist_merged

In [ ]:
# Genes analyzed individually
genes_of_interest = config.require('chip', 'genes_of_interest')

In [ ]:
# Keep cases and controls, add each participant's CHIP gene to the merged table and
# remove duplicate calls
wl_final_anno_df = wl_final_anno_df.loc[wl_final_anno_df[status_col].isin([case_label, control_label])]
wl_final_merged_df = wl_final_merged_df.loc[wl_final_merged_df[status_col].isin([case_label, control_label])]

print("Adding the CHIP gene of each participant and removing duplicate calls:")
wl_final_merged_df, wl_final_anno_df = chip_funcs.add_chip_gene_and_deduplicate(
    wl_final_merged_df,
    wl_final_anno_df
)

In [ ]:
# Plot variant types for the most frequently mutated CHIP genes
print("Figure: variant types by gene for the 30 most frequently mutated CHIP genes")
variant_plot_data, variant_colors = chip_plots_funcs.create_chip_variants_plot(
    wl_final_anno_df,
    top_n=30,
    output_filename=None
)

In [ ]:
# Plot CHIP variants in the genes of interest by disease group
print("Figure: CHIP variants in the genes of interest by disease group, as counts and "
      "proportions")
counts_df, proportions_df = chip_plots_funcs.plot_chip_variants_by_statustype(
    wl_final_anno_df,
    genes_of_interest=genes_of_interest,
    output_filename=None
)

In [ ]:
# Test whether the proportion of CHIP variants in each gene differs between disease groups
print("Comparison of CHIP gene proportions between disease groups:")
proportion_results = chip_funcs.analyze_chip_gene_proportion_differences(
    wl_final_anno_df, 
    genes_of_interest=genes_of_interest
)

In [ ]:
# Gene VAF analysis and plot
print("Comparing VAF between the genes of interest:")
gene_vaf_stats = chip_funcs.analyze_vaf_distribution_differences(
    wl_final_anno_df,
    analysis_type='gene',
    genes_of_interest=genes_of_interest
)

print("Figure: VAF distribution for each gene of interest and for all CHIP variants")
fig, ax, plot_data = chip_plots_funcs.plot_gene_vaf_violin(
    wl_final_anno_df,
    genes_of_interest=genes_of_interest,
    stats_df=gene_vaf_stats['stats_df'],
    output_filename=None
)


In [ ]:
# TET2 subtype VAF analysis and plot
print("Comparing TET2 VAF between disease groups:")
tet2_vaf_stats = chip_funcs.analyze_vaf_distribution_differences(
    wl_final_anno_df,
    analysis_type='subtype',
    gene_for_subtype='TET2'
)

print("Figure: TET2 VAF distribution by disease group")
fig, ax, subtype_data = chip_plots_funcs.plot_tet2_vaf_by_subtype_violin(
    wl_final_anno_df,
    stats_df=tet2_vaf_stats['stats_df'],
    output_filename=None
)


In [ ]:
# Compare the age of carriers of each gene of interest between disease groups, and plot it
print("Comparing the age of carriers of each gene of interest between disease groups:")
gene_age_stats = chip_funcs.analyze_gene_age_by_subtype(
    wl_final_anno_df,
    genes=genes_of_interest
)


print("Figure: age of carriers of each gene of interest by disease group")
fig, axes = chip_plots_funcs.plot_gene_age_swarm(
    wl_final_anno_df,
    stats_results=gene_age_stats,
    genes=genes_of_interest,
    output_filename=None
)

In [ ]:
# Plot the prevalence of recurrent variants in the genes of interest across disease groups
print("Figure: prevalence of recurrent variants (seen at least twice, up to 10 per gene) in "
      "the genes of interest, by disease group")
fig, ax, heatmap_data, variant_info = chip_plots_funcs.plot_variant_prevalence_heatmap(
    wl_final_anno_df,
    genes_of_interest=genes_of_interest,
    min_variant_count=2,
    top_n_per_gene=10,
    output_filename=None
)